# Notebook 2: Portfolio Summary (Easy - SQL + Python)
Combine SQL queries with Python pandas for portfolio breakdown and formatted output.

### Fetch All Loan Details with Customer Info
This query joins the Loans and Customers tables to pull a complete view of every loan along with the borrower's name, credit score, annual income, and state. The results are ordered by loan amount (largest first) and stored for use in the Python cells below.

In [ ]:
%%sql -r loan_details
SELECT l.LOAN_ID, l.LOAN_TYPE, l.LOAN_AMOUNT, l.INTEREST_RATE, l.TERM_MONTHS,
       l.LOAN_STATUS, l.PURPOSE, l.MONTHLY_PAYMENT,
       c.FIRST_NAME || ' ' || c.LAST_NAME AS CUSTOMER_NAME,
       c.CREDIT_SCORE, c.ANNUAL_INCOME, c.STATE
FROM LOAN_DB.LENDING.LOANS l
JOIN LOAN_DB.LENDING.CUSTOMERS c ON l.CUSTOMER_ID = c.CUSTOMER_ID
ORDER BY l.LOAN_AMOUNT DESC

### Quick Portfolio Stats
Using the data fetched above, this cell computes a few headline numbers — total loan count, combined portfolio value, average loan size, number of distinct loan types, and how many states are represented. It gives a quick sanity check before diving into deeper analysis.

In [ ]:
import pandas as pd

df = loan_details.copy()
print(f"Total Loans: {len(df)}")
print(f"Total Portfolio: ${df['LOAN_AMOUNT'].sum():,.2f}")
print(f"Average Loan: ${df['LOAN_AMOUNT'].mean():,.2f}")
print(f"\nLoan Types: {df['LOAN_TYPE'].nunique()}")
print(f"States Covered: {df['STATE'].nunique()}")

### Portfolio Breakdown by Loan Type
This cell groups loans by type and aggregates key metrics — loan count, total amount, average interest rate, average credit score, and each type's percentage share of the total portfolio. Results are sorted by total amount to highlight the most significant segments.

In [ ]:
summary = df.groupby('LOAN_TYPE').agg(
    count=('LOAN_ID', 'count'),
    total_amount=('LOAN_AMOUNT', 'sum'),
    avg_rate=('INTEREST_RATE', 'mean'),
    avg_credit_score=('CREDIT_SCORE', 'mean')
).round(2)

summary['pct_of_portfolio'] = (summary['total_amount'] / summary['total_amount'].sum() * 100).round(1)
summary = summary.sort_values('total_amount', ascending=False)
summary

### Loan Amount by Type and Status
This cell creates a pivot table that cross-references loan type against loan status, showing the total dollar exposure in each combination. A TOTAL column is added to rank loan types by overall size. This helps identify which types carry the most risk based on their status distribution.

In [ ]:
status_pivot = df.pivot_table(
    index='LOAN_TYPE',
    columns='LOAN_STATUS',
    values='LOAN_AMOUNT',
    aggfunc='sum',
    fill_value=0
)
status_pivot['TOTAL'] = status_pivot.sum(axis=1)
status_pivot = status_pivot.sort_values('TOTAL', ascending=False)
status_pivot

### Loan Size Distribution Buckets
This cell segments loans into five size bands — under 10K, 10K–50K, 50K–100K, 100K–250K, and 250K+ — and summarises the count, total amount, and average interest rate within each band. This reveals how loan sizes are distributed across the portfolio and whether pricing varies by loan size.

In [ ]:
bins = [0, 10000, 50000, 100000, 250000, 700000]
labels = ['<10K', '10K-50K', '50K-100K', '100K-250K', '250K+']
df['SIZE_BUCKET'] = pd.cut(df['LOAN_AMOUNT'], bins=bins, labels=labels)

bucket_summary = df.groupby('SIZE_BUCKET', observed=True).agg(
    count=('LOAN_ID', 'count'),
    total=('LOAN_AMOUNT', 'sum'),
    avg_rate=('INTEREST_RATE', 'mean')
).round(2)
bucket_summary